In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


In [3]:
# 1. Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


Using device: cpu


In [4]:
# 2. Load MNIST Dataset
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)



100%|██████████| 9.91M/9.91M [00:00<00:00, 58.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.72MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.15MB/s]


In [5]:
# 3. Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)


In [6]:
# 4. Create Neural Network
class MNISTModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.network = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)


# Create model
model = MNISTModel().to(device)

print(model)


MNISTModel(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)


In [7]:
# 5. Loss Function
loss_fn = nn.CrossEntropyLoss()

In [8]:
# 6. Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [9]:
# 7. Training
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        # Move data to device
        images = images.to(device)
        labels = labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = loss_fn(outputs, labels)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        # Loss
        total_loss += loss.item()

        # Accuracy
        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {total_loss / len(train_loader):.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )

Epoch [1/5] Loss: 0.3411 Accuracy: 90.36%
Epoch [2/5] Loss: 0.1382 Accuracy: 95.83%
Epoch [3/5] Loss: 0.0954 Accuracy: 97.09%
Epoch [4/5] Loss: 0.0724 Accuracy: 97.80%
Epoch [5/5] Loss: 0.0570 Accuracy: 98.28%


In [10]:
# 8. Test Model
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)


test_accuracy = 100 * correct / total

print(
    f"\nTest Accuracy: {test_accuracy:.2f}%"
)


Test Accuracy: 97.34%


In [11]:
# 9. Test One Image
image, label = test_dataset[0]

image = image.unsqueeze(0).to(device)

with torch.no_grad():

    output = model(image)

    prediction = output.argmax(dim=1).item()

print("Actual Digit:   ", label)
print("Predicted Digit:", prediction)

Actual Digit:    7
Predicted Digit: 7
